# Seating Six Feet Apart

**Reading:** Chapter 14, Modeling with Integer Programs

### Introduction

As campuses reopen for the fall semester, implementing social distancing measures is a top priority. Students need to maintain a 6 foot distance in lecture halls which means the seating capacity will drop. This is an issue for large and small classes when it comes to room assignments. For ENGRI 1101, we want to enable as many students as possible to attend in-person labs so that they can get immediate help or use the lab computers.  

In [ ]:
# imports -- make sure you run this cell
import pandas as pd
import numpy as np
import math, itertools
import matplotlib.pyplot as plt
import networkx as nx
from ortools.linear_solver import pywraplp as OR
import shapely
from shapely.geometry import Polygon, Point

**Names of group members:**

**Who will be "driving" for the first portion of this lab: (write their name and NetID here)**

_As a reminder, only the person driving is supposed to touch the keyboard. Everyone in the group must read the lab, help answer questions, etc. Also, the driver must still discuss/explain/etc question answers with their group—they cannot just write down answers without discussion even if they know them._

## Part 1: Brainstorming

### Questions

- How can we decide which seats will be used so that the most number of seats are available without breaking social distancing rules?
- How do we turn a classroom layout into something we can solve?

A good strategy for tackling complex problems such as this is to work through small examples. Let's get started!

**Example 1**
Given the following rows of seats, think about how it could be represented as a graph.

In [ ]:
ex1 = plt.imread('images-lab/ex1.png')
plt.axis('off')
plt.imshow(ex1);

**Q1.1:** What are the nodes?

**A:**

_Write your answer here._

    

**Q1.2:** What are the edges? (Hint: What relationship b/w the nodes do we care about? Hint 2: Related to distance, but the edges do not have distance attributes.)

**A:**

_Write your answer here._

Let's visualize the graph by adding the specific nodes and edges you previously described to the lists V and E. Assume that a chair's 6-foot radius includes 2 chairs to either side, front, back, and the diagonals. For example, if someone occupies seat 7, no one can occupy 5, 6, 8, 3, 11, 2, 4, 10, and 12.

**Q1.3:** Assume someone occupies seat 3. Which seats cannot be occupied?

**A:**

_Write your answer here._

In [ ]:
from seat_packing_lab import ex1

V = [1,2,3,4,5,6,7,8,9,10,11,12]
E = [(1,2),(1,3),(1,5),(1,6),(2,3),(2,4),(2,5),(2,6),(2,7),(3,4),(3,6),(3,7),(3,8),(4,7),(4,8),
     (5,6),(5,7),(5,9),(5,10),(6,7),(6,8),(6,9),(6,10),(6,11),(7,8),(7,10),(7,11),(7,12),(8,11),(8,12),
     (9,10),(9,11),(10,11),(10,12),(11,12)]

ex1(V, E)

**Q1.4:** Give a feasible solution for this graph. It does not have to be optimal.

**A:**

_Write your answer here._

**Q1.5:** In general, what will a feasible solution look like?

**A:**

_Write your answer here._

A set of nonadjacent nodes is called an independent set. We will now look for an independent set of maximum cardinality.

## Part 2: Solving

To reiterate, an independent set must only contain nodes that are not adjacent to each other; if nodes joined by an edge are in the same set, then the set is not independent. The solution we want is not just any independent set--we want the one with the maximum cardinality or largest in size. It is possible for there to be more than one *maximum independent set* (MIS). This type of problem, also called maximum independent set, can be written as an integer program. Think about what constraints should be in the integer program to give us the optimal solution while working through the next example.

**Example 2** Find a MIS.

In [ ]:
from seat_packing_lab import ex2
V2 = [1,2,3,4,5,6]
E2 = [(1,2),(2,3),(1,4),(4,2),(2,5),(5,3),(4,6),(5,6)]
ex2(V2, E2)

**Q2.1:** Which nodes are in the maximum independent set? (This example has a unique MIS.)

**A:**

_Write your answer here._

**Q2.2:** What are some strategies you tried or patterns you noticed when getting to the answer?

**A:**

_Write your answer here._

For the edge (1, 2) either 1 or 2 or neither are in the MIS. For the triangle formed by nodes 1, 2, and 4, at most one of the three are in the MIS. The edge example suggests a constraint like $x_1 + x_2 \leq 1$ where $x_1$ = {0 if node 1 not in MIS, 1 if in MIS} and $x_2$ likewise for node 2. The triangle constraint would be $x_1 + x_2 + x_4 \leq 1$.

**Q2.3:** Is the triangle constraint necessary or redundant?

**A:**

_Write your answer here._

    

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

Now, you are ready to write out the integer program! This OR-Tools model takes an adjacency matrix as an input. An adjacency matrix is a way of representing a graph as a matrix. Here, the entry at $(i,j)$ is 1 if $i$ and $j$ share an edge and 0 otherwise. Below, we load an adjacency matrix representing the 12 seat example.

In [ ]:
mis_seat_packing_example = pd.read_csv('data/mis_seat_packing_example.csv',index_col=0).astype(int)
mis_seat_packing_example.columns = mis_seat_packing_example.columns.astype(int)
display(mis_seat_packing_example)

**Q2.4:** Finish defining the model below

**A:**

_Write your answer here._

In [ ]:
def mis(graph, integer=False):
    """A model for solving the maximum independent set problem.
    
    Args:
        graph (pd.DataFrame): Graph represented by an adjacency matrix.
    """
    NODES = list(graph.index)                # nodes
    EDGES = []                               # edges
    for i in NODES:
        for j in NODES:
            if i <= j and graph.at[i,j] == 1:
                EDGES.append((i,j))
    
    # define model
    m = OR.Solver('mis', OR.Solver.CBC_MIXED_INTEGER_PROGRAMMING)
    
    # decision variables
    x = {} 
    for i in NODES:
        if integer:
            x[i] = m.IntVar(0, 1, ('%s' % (i)))
        else:
            x[i] = m.NumVar(0, 1, ('%s' % (i)))
    
    # objective function
    m.Maximize(sum(x[i] for i in NODES))
        
    # subject to: no vertices in the set share an edge (i.e., for each edge (i,j), at most one of i and j can be in the set)
    # TODO: implement this constraint
    # Remember that to add a constraint, we can use m.Add(CONSTRAINT_EXPRESSION)
    # Hint: recall that EDGES is a list of tuples (i,j) where vertices i and j share an edge. How did we implement a loop for defining the decision variables?
    
    

    return m,x

In [ ]:
def solve(m):
    m.Solve()
    print('Solution:')
    print('Objective value =', m.Objective().Value())
    return {var.name() : var.solution_value() for var in m.variables()}

Run this cell to find an optimal solution to our first example!

In [ ]:
m,x = mis(mis_seat_packing_example, integer=True)
solve(m)

**Q2.5:** What was the optimal solution?

**A:**

_Write your answer here._

**NOTE:** A clique is a set of nodes such that an edge exists for any pair of nodes in the clique. In example 1, {1,2,5,6}, {5,6,9,10}, {3,4,7,8}, and {7,8,11,12} are all the cliques. The union of them is the entire node set. For any clique, at most one node can be picked to be in an independent set; therefore, we have an upper bound of 4 on the cardinality of any independent set in example 1.

Now, let's solve the second example! First, we load the adjacency matrix.

In [ ]:
mis_small_example = pd.read_csv('data/mis_small_example.csv',index_col=0).astype(int)
mis_small_example.columns = mis_small_example.columns.astype(int)
display(mis_small_example)

In [ ]:
m,x = mis(mis_small_example, integer=True)
solve(m)

## Part 3: Solving the Actual Data

Let's return to the original seat packing problem. 
We look at a few examples. Here is a classroom where no pair of students may sit less than 6' apart (as was true during the pandemic). The circles have radii of 6', and the yellow boxes indicate possible seats for students.

<img src="images-lab/classroom-seat-count.jpg" height="600">

**Q3.1:** How many students can we seat in this classroom? We can only use the yellow seats, and no two students can sit within 6' (the radii of one circle) of each other.

**A:**

_Write your answer here._

That's a lot! Let's look at a smaller classroom that only seats 30 students.

In [ ]:
room = plt.imread('images-lab/floorplan.jpg')
plt.subplots(dpi=300)
plt.axis('off')
plt.imshow(room);

The following is the same room layout with a few extra things drawn on top. (The drawings are done using the shapely package installed at the beginning. For the exact code, check out the ex_room() function in *seats_lab.py*.)

In [ ]:
from seat_packing_lab import ex_room
polys, points = ex_room()

**Q3.2:** What do the blue squares and circles represent?

**A:**

_Write your answer here._

**Q3.3:** What do you think the yellow bar at the top is telling us?

**A:**

_Write your answer here._

**Q3.4:** If each chair center is a node in our graph, when should there be an edge between node (chair) $i$ and node $j$?

**A:**

_Write your answer here._

In practice, we want to be a little more conservative. We don't just want the distance between chair centers to be at least 6'; the guideline is that the distance between the central point of the original chair and the *boundary* of a potential 'neighbor' chair is no more than 6 feet.

A brief overview:
- The layout image is imported.
- We find the location of each chair (manually here but can also use computer vision).
- Distances are in pixels, so we calculate the scale using the orange bar which we know is 10 feet as a reference.
- Blue chairs and points are plotted onto the image.

**Q3.5:** The code below transforms our data into a graph, we determine the edges by making a list of neighbors for each node. Complete the code below by adding a check (Boolean condition) for when we should add an edge to our graph. Note that the distances in our graph are in units of pixels, and we estimate 1 foot to be about $14 \frac{1}{6}$ pixels.

**A:**

_Write your answer here._

**A:** *Do not write your answer here. Modify the code below.*

In [ ]:
# (code by Sander Aarts)

# define a dataframe of Polygons and Points
df = pd.DataFrame(list(zip(polys, points)), 
               columns =['polygon', 'point'])

# generate edges from distances
df['neighbors'] = None # list of neighbors for each node

# populate an adjacency matrix
room_233 = np.zeros((len(df),len(df)))

for i in range(df.shape[0]):
    neighbors = list()     # get empty list
    for j in range(df.shape[0]):
        if (i != j):
            dist = df['polygon'][j].distance(df['point'][i]) # distance from end of node j to center of node i, in pixels
            # CHANGE CODE BELOW
            if True:  # TODO: change the "True" to check if the distance is less than 6 feet (don't forget to convert feet to pixels!)
            # END CODE CHANGE
                neighbors.append(df.index[j])  # add j to the list of neighbors for i (i.e., add (i,j) as an edge in the graph)
                room_233[i,j] = 1
    if (len(neighbors) > 0):
        df['neighbors'][i] = neighbors
room_233 = pd.DataFrame(room_233)

With the graph ready, it is time to use the integer program we wrote previously.

In [ ]:
m,x = mis(room_233, integer=True)
sol = solve(m)

**Q3.6:** How many chairs can we fit in this classroom according to the IP? How does this compare to our previous solution?

**A:**

_Write your answer here._

This is about a 6.66\% improvement from the previous solution!

We can update our dataframe and see what the solution looks like after its "Cinderella" moment.

In [ ]:
from seat_packing_lab import ex_room_sol
ex_room_sol(df, sol)

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

## Part 4: Changing the Granularity

It was mentioned in the introduction that ENGRI 1101 wanted to maximize lab classroom seating. Unlike in lecture halls, the chairs have wheels and can be moved. How will you approach this problem?

In [ ]:
# show Rhodes 571
room = plt.imread('images-lab/labclassroom.png')
plt.subplots(dpi=300)
plt.axis('off')
plt.imshow(room);

Imagine that all the chairs in Rhodes 571 are taken outside the room and brought back one at a time. The tables are fixed in place. (Ignore the laptops since they are movable.) Where can you place each chair that is brought back?

**Q4.1:** Will there be fewer or more nodes than chairs currently in the image?

**A:**

_Write your answer here._

**Q4.2:** Describe where there might be nodes. Are they close or far apart?

**A:**

_Write your answer here._

In [ ]:
# add drawings
from seat_packing_lab import ex_lab
polys2, points2 = ex_lab()

Is this what you pictured? Each rectangle represents a possible chair placement. The distance between these possible chair placements is arbitrary. In reality, there are an infinite number of chair placements but then the problem would be too big. The arbitrary distance used was a good balance between accuracy and the problem size. Run the next cell to see how many nodes and edges we have here.

In [ ]:
# (code by Sander Aarts)

# define a dataframe of Polygons and Points
df2 = pd.DataFrame(list(zip(polys2, points2)), 
               columns =['polygon', 'point'])

# populate an adjacency matrix
room_571 = np.zeros((len(df2),len(df2)))

for i in range(df2.shape[0]):
    for j in range(df2.shape[0]):
        if (i != j):
            dist = df2['polygon'][j].distance(df2['point'][i])
            if (dist <= 51):    # 51 pixels = 6 feet
                room_571[i,j] = 1
room_571 = pd.DataFrame(room_571)

(Solving this integer program takes around 30 seconds)

In [ ]:
m,x = mis(room_571, integer=True)
sol2 = solve(m)

22 is the capacity if chairs are not moved. 29 is a **31.8%** increase!

**Q4.3:** Why do you think it took so much longer for the solver to find the solution to this problem?

**A:**

_Write your answer here._

In [ ]:
print('There are %d nodes and %d edges.' % (df2.shape[0], room_571.sum().sum()))

Our graph is much larger. Run the cell below to see exactly how big it is. Even at this scale, it's impressive that our solvers can solve the problem! 

In [ ]:
# show solution on room
from seat_packing_lab import ex_lab_sol
ex_lab_sol(df2, sol2)

## Part 5: Clustering

The same integer programming ideas that helped us choose seats can also help us make sense of data. In the Minimum Spanning Tree lab, we used New York City taxi pickup data from 2014. Each day of the year was represented by 96 numbers: one pickup count for each 15-minute interval.

Our goal in this part is to group days into at most 3 clusters using a **k-medoid** model. The letter $k$ means the maximum number of centers we are allowed to open, so here $k=3$.

In k-medoid clustering, each cluster center, or medoid, must be one of the real days in the data. The model chooses up to 3 real days as centers, then assigns every day to one selected center. Let's take a look at the data and remind ourselves what it looks like.

In [ ]:
import pickle
from pathlib import Path
from IPython.display import Image
from clustering import *

taxi_data_path = Path('../minimum_spanning_tree/data/taxi_count_dict.pickle')
with open(taxi_data_path, 'rb') as handle:
    taxi_counts = pickle.load(handle)

taxi_days = pd.DataFrame(taxi_counts)
taxi_days['total_pickups'] = taxi_days['count_vector'].apply(sum)
taxi_days['date'] = taxi_days['m'].astype(str) + '/' + taxi_days['d'].astype(str)
taxi_days.head(7)


The `weekday` column uses 0 for Monday, 1 for Tuesday, and so on. The MST lab visualized this same data as a calendar of daily pickup curves:

In [ ]:
Image('../minimum_spanning_tree/images/taxi_ride_frequency.png', width=1000)


Before we write an IP, we need a way to measure distance between two real days.

The original data has 96 values per day. To keep the model smaller, we will group those into two-hour pickup counts. 

**Q5.1:** In the code block below, determine how many time blocks we have per day, and set `K` equal to the number of clusters that we want.

**A:** _Do not write your answer here. Modify the code block below._

In [ ]:
# Start small so the IP solves quickly and the output is easy to read.
# Later, try list(range(21)) or list(range(28)).
DAYS = list(range(14))  # first 2 weeks of 2014

# CHANGE YOUR CODE HERE
TIME_BLOCKS = list(range(0))  # TODO: change 0 to the correct number of two-hour time blocks in a day
K = 0  # TODO: change 0 to the correct number of centers to use in the IP model
# END CODE CHANGE HERE

features = {i: two_hour_day_vector(taxi_days, i) for i in DAYS}

display(taxi_days.loc[DAYS, ['date', 'weekday', 'total_pickups']].head(10))
print('Using %d days, at most %d centers, and %d raw two-hour pickup counts per day.' % (len(DAYS), K, len(TIME_BLOCKS)))



For k-medoid, we will use **absolute distance**. If one day has value 80 at 8 AM and another day has value 60 at 8 AM, the 8 AM difference is $|80 - 60| = 20$. We add these differences over all 12 two-hour blocks. That total becomes the fixed cost $c_{ij}$ of assigning day $j$ to center day $i$. Take a look at the pickups on January 1 and January 2, rounded to the nearest thousand.

In [ ]:

comparison_df = pd.DataFrame({
    'Time Block': [time_block_label(t) for t in TIME_BLOCKS],
    f"Day {DAYS[0]+1} Pickups (1000s)": np.round(features[DAYS[0]]/1000).astype(int),
    f"Day {DAYS[1]+1} Pickups (1000s)": np.round(features[DAYS[1]]/1000).astype(int),
})

display(comparison_df)

**Q5.2:** The distance between two days is the sum of the absolute differences in their 12 two-hour pickup counts. Compute the distance between the first two days in `DAYS`. In other words, compute the distance between day `DAYS[0]` and day `DAYS[1]`.

**A:**

_Write your answer here, then run the code below to check it._


In [ ]:
print(np.abs(np.round(features[0]/1000) - np.round(features[1]/1000)).sum())


What if we didn't round? What would the distance be? Take a look and make sure you understand what this code is doing!

In [ ]:
print(np.abs(features[0] - features[1]).sum())  # features[i] gives us a list of the two-hour pickup counts for day i

In the same way you just calculated the distance (rounded) between January 1st and January 2nd, we need to do the same for all days in our dataset. We compute all pairwise assignment costs $c_{ij}$, the distance from possible center day $i$ to assigned day $j$, based on the data.

**Q5.3:** Complete the code cell below by filling out the `day_distance` function. Hint: reference the code cell right above this!

**A:** _Do not write your answer here. Fill out the code below._

In [ ]:
def day_distance(i, j, features):
    '''Given two days i and j, return the distance between them as the sum of the absolute differences in their two-hour pickup counts.'''
    return None  # TODO: implement this function

cost = {(i, j): day_distance(i, j, features) for i in DAYS for j in DAYS}

cost_table = pd.DataFrame(index=DAYS, columns=DAYS)
for i in DAYS:
    for j in DAYS:
        cost_table.at[i, j] = cost[i, j]

display(cost_table)


### Building The Decisions

Let $N$ be the set of days. We will choose up to $k$ center days from this same set.

Decision variables:

- $y_i = 1$ if day $i$ is chosen as a center, and $0$ otherwise.
- $x_{ij} = 1$ if day $j$ is assigned to center day $i$, and $0$ otherwise.

Data:

- $c_{ij}$ is the fixed distance cost of assigning day $j$ to center day $i$.

The objective is:

$$\min \sum_{i \in N}\sum_{j \in N} c_{ij}x_{ij}$$

**Q5.4:** If we are allowed to choose at most $k$ centers, what other constraint should we add?

**A:**

_Write your answer here._


**Q5.5:** For a fixed assigned day $j$, which expression says that day $j$ is assigned to exactly one center?

**A:**

_Write your answer here._


**Q5.6:** If our goal is to minimize the total distance between all days and the center they are assigned to, what should our objective function be?

**A:**

_Write your answer here._


A lot of integer programs need rules that sound like this:

> If one binary variable is 1, then another binary variable must also be 1.

In our case, we want something that says "day $j$ can only be assigned to center $i$ if $i$ is assigned as a center.

Let's start with a small example to see how we might write a constraint to that end. Suppose:

- $x = 1$ means "we assign a student to a lab table."
- $y = 1$ means "that lab table is open."

The rule is: **if a student is assigned to the table, then the table must be open.**

**Q5.7:** Fill in the truth table below. Which row is the only row that should be forbidden?

**A:** _Fill in the truth table below._

| $x$ | $y$ | Meaning | Should this be allowed? |
|---|---|---|---|
| 0 | 0 | no student assigned, table not open | |
| 0 | 1 | no student assigned, table open | |
| 1 | 0 | student assigned, table not open | |
| 1 | 1 | student assigned, table open | |

**Q5.8:** Before reading further, try to invent a linear inequality using only $x$ and $y$ that rules out the bad row but allows the other three rows. Your inequality can use symbols like $\leq$, $\geq$, $+$, and numbers.

**A:**

_Write your answer here._

**Q5.9:** Test your inequality in the table below to check that it exactly allows the three good rows and forbids the one bad row.

**A:**

_Fill out the table and answer the question below._
| $x$ | $y$ | Should this be allowed? | Does your inequality allow it? |
|---|---|---|---|
| 0 | 0 | | |
| 0 | 1 | | |
| 1 | 0 | | |
| 1 | 1 | | |

Did the inequality work? - ??

The trick we typically use is to constrain

$$x \leq y.$$

This inequality captures the rule **if $x=1$, then $y=1$** because:

| $x$ | $y$ | Is $x \leq y$ true? |
|---|---|---|
| 0 | 0 | yes |
| 0 | 1 | yes |
| 1 | 0 | no |
| 1 | 1 | yes |

In this k-medoid model, the same idea appears as:

$$x_{ij} \leq y_i$$

This says day $j$ can be assigned to center day $i$ only if day $i$ is actually opened as a center. (Verify for yourself that this constraint works!)

Here is the full IP formulation:
\begin{align*}
\min & \sum_{i \in N}\sum_{j \in N} c_{ij}x_{ij} \\
\sum_{i \in N} x_{ij} &= 1 \quad \text{for every assigned day } j \in N \\
x_{ij} &\leq y_i \quad \text{for every } i,j \in N \\
\sum_{i \in N} y_i &\leq k \\
x_{ij} &\in \{0,1\} \quad \text{for every } i, j \in N \\
y_i &\in \{0,1\} \quad \text{for every i} \in N
\end{align*}

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

**Q5.10:** Why do we still have a linear program? That is, in the objective function, why is it not a problem that we multiply $c_{ij}$ with $x_{ij}$?

**A:** _Write your answer here._

**Q5.11:** Why does minimizing $\sum_{i \in N}\sum_{j \in N} c_{ij}x_{ij}$ encourage days to be assigned to nearby centers?

**A:**

_Write your answer here._


The next cell gives starter code for the k-medoid IP. 

**Q5.12:** Complete the code below by finishing the assignment constraints, the open-center constraints, the center-count constraint, and the objective.

**A:** _Do not write your answer here. Fill in the code below._


In [ ]:
def taxi_kmedoid(days, k, cost):
    """Build a k-medoid IP using fixed pairwise assignment costs."""
    m = OR.Solver('taxi_kmedoid', OR.Solver.CBC_MIXED_INTEGER_PROGRAMMING)
    m.SetTimeLimit(60000)  # 60 seconds maximum

    # x[i, j] = 1 if day j is assigned to center day i.
    x = {(i, j): m.IntVar(0, 1, 'x_%s_%s' % (i, j)) for i in days for j in days}

    # y[i] = 1 if day i is chosen as a center.
    y = {i: m.IntVar(0, 1, 'y_%s' % i) for i in days}

    # Constraint 1: every day j is assigned to exactly one center i.
    for j in days:
        # TODO: finish this line.
        # m.Add(sum(x[i, j] for i in days) == ???)
        pass

    # Constraint 2: day j can be assigned to center i only if center i is open.
    for i in days:
        for j in days:
            # TODO: finish this line.
            # m.Add(x[i, j] <= ???)
            pass

    # Constraint 3: choose at most k centers.
    # TODO: finish this line.
    # m.Add(sum(y[i] for i in days) <= ???)

    total_cost = sum(cost[i, j] * x[i, j] for i in days for j in days)

    # Objective: minimize total assignment cost.
    # TODO: finish this line.
    # m.Minimize(???)

    raise NotImplementedError('Finish the TODOs in taxi_kmedoid, then delete this line.')

    return m, x, y


We've imported some helper functions to solve the k-medoid model and summarize the result. You do not need to edit them. Read the output like this:

- **Center day:** the real day chosen as the center of a cluster.
- **Size:** how many days were assigned to that center.
- **Weekday counts:** whether the assigned days are mostly weekdays, weekends, or mixed.
- **Average distance / maximum distance:** how close the assigned days are to the center day.
- **Peak hour:** the hour when the center day has the largest pickup count.

Run the cell below to see the solution to our taxi clustering problem!

In [ ]:
m, x, y = taxi_kmedoid(DAYS, K, cost)
kmedoid_solution = solve_and_analyze_kmedoid('K-medoid with selected real-day centers',
                                             m, x, y, DAYS, TIME_BLOCKS, cost, features, taxi_days)


**Q5.13:** Which days were chosen as centers? Do they seem like reasonable representative days?

**A:**

_Write your answer here._

**Q5.14:** Look at the cluster sizes and weekday counts. Did the model mostly separate weekdays from weekends? Are there any days that surprise you?

**A:**

_Write your answer here._

**Q5.15:** Look at the center-day curves in the plot. How are the chosen representative days different from each other?

**A:**

_Write your answer here._

**Q5.16:** Look at the `avg_distance` and `max_distance` columns. Is every day close to its center, or are there some days that fit their cluster poorly?

**A:**

_Write your answer here._

Each day is a point in 12 dimensions, one coordinate per two-hour block, which is impossible to draw directly. We can use an algorithm called **Principal Component Analysis (PCA)** to find the two directions along which the days differ the most and flatten the data onto those two directions, keeping as much of the spread as it can. We can interpret points that are close together in the plot as days that had similar pickup curves. Notice that there are 14 points, one for each of our 14 days, colored by the cluster the IP assigned it to, and the stars are the center days.


In [ ]:
plot_kmedoid_pca(kmedoid_solution['clusters'], features,
                 'First 14 days of 2014, projected onto two principal components',
                 dates=taxi_days['date'].to_dict())


**Q5.17:** What do you notice in the scatterplot? Are the clusters well separated, or do they blend into each other? Where do the center days sit relative to the days assigned to them?

**A:**

_Write your answer here._


### Clustering The Entire Dataset

So far, we used only the first 14 days so the model would solve quickly and the output would be easy to inspect. Now try the same k-medoid model on every day in the dataset (it'll take longer)!

In [ ]:
ALL_DAYS = list(taxi_days.index)
all_features = {i: two_hour_day_vector(taxi_days, i) for i in ALL_DAYS}
all_cost = {(i, j): day_distance(i, j, all_features) for i in ALL_DAYS for j in ALL_DAYS}

m, x, y = taxi_kmedoid(ALL_DAYS, K, all_cost)
full_year_solution = solve_and_analyze_kmedoid('K-medoid with selected real-day centers, full dataset',
                                               m, x, y, ALL_DAYS, TIME_BLOCKS, all_cost, all_features, taxi_days)


**Q5.18:** Compare the full-dataset clusters to the two-week clusters. Is the overall pattern still similar, such as weekday-like days versus weekend-like or late-night days? What changes when the model sees the whole year?

**A:**

_Write your answer here._


Now the same picture for the whole year. Each point is one day of 2014, colored by the cluster the IP assigned it to, and the stars are the center days.


In [ ]:
plot_kmedoid_pca(full_year_solution['clusters'], all_features,
                 'Every day of 2014, projected onto two principal components',
                 dates=taxi_days['date'].to_dict())


**Q5.19:** What do you notice in the scatterplot? Are the clusters well separated, or do they blend into each other? Where do the center days sit relative to the days assigned to them, and are there any points that look like they were assigned to the "wrong" cluster? Do you notice any outliers, and if so, what do you think they might be?

**A:**

_Write your answer here._


### Key Takeaways

Congratulations! You just got a taste of the actual process used to determine seating configurations subject to 6-feet social distancing requirements for Cornell (and it was done by previous ENGRI 1101 students too!). 

In this lab, you modeled a social distancing seating problem using ideas from graph theory and optimization. You saw how to represent possible seats as nodes in a graph and connect seats with an edge when they are too close to be used at the same time. This turned the seating problem into a maximum independent set problem, where the goal is to choose as many nodes as possible without selecting adjacent ones.

You also used integer programming to solve this problem computationally. Along the way, you connected the meaning of the graph to the model constraints, interpreted solutions in both small examples and real classroom data, and saw how modeling choices affect problem size. In particular, you learned that increasing the granularity of possible seat placements can make the model more realistic (and give a better solution), but it also makes the optimization problem much harder to solve.

In Part 5, you also saw how the same modeling ingredients can be used beyond seat packing. For k-medoid clustering, the model chooses representative days as centers, assigns every day to one selected center, and minimizes the total assignment cost. The distances $c_{ij}$ are fixed data, while the binary variables $x_{ij}$ and $y_i$ represent the decisions.